# Bài tập Phân loại Hoa bằng Mạng Nơ-ron Tích chập (CNN)

Trong bài tập này, bạn sẽ:

- Hiểu cơ chế hoạt động của **Convolutional Neural Network (CNN)**.
- Xây dựng kiến trúc CNN từ đầu bằng **Keras / TensorFlow**.
- **Tiền xử lý** dữ liệu ảnh hoa 7 lớp.
- **Huấn luyện** mô hình với EarlyStopping và ModelCheckpoint.
- **Đánh giá** độ chính xác, visualize ma trận nhầm lẫn và feature maps.

## 1. Cài đặt môi trường và import thư viện

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0)

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 2. Tải và tiền xử lý dữ liệu

Bộ dữ liệu gồm **7 lớp hoa**: bellflower, daisy, dandelion, lotus, rose, sunflower, tulip.

**Các bước tiền xử lý:**
1. Đọc ảnh bằng OpenCV (BGR)
2. Chuyển BGR → RGB (Keras/TF dùng RGB)
3. Resize về `64×64`
4. Normalize pixel về `[0.0, 1.0]` bằng cách chia `/255`

In [ ]:
DATASET_DIR = 'flower-training'
IMG_SIZE    = (64, 64)
CLASSES     = ['bellflower', 'daisy', 'dandelion', 'lotus', 'rose', 'sunflower', 'tulip']

def load_dataset(dataset_dir):
    X, y = [], []
    for label_idx, cls in enumerate(CLASSES):
        cls_dir = os.path.join(dataset_dir, cls)
        if not os.path.exists(cls_dir):
            print(f'Bỏ qua lớp thiếu: {cls_dir}')
            continue
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
                continue
            img = cv2.imread(os.path.join(cls_dir, fname))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # BGR → RGB
            img = cv2.resize(img, IMG_SIZE)               # resize 64x64
            X.append(img.astype(np.float32) / 255.0)     # normalize /255
            y.append(label_idx)
    return np.array(X), np.array(y)

print('Đang tải dataset...')
X, y = load_dataset(DATASET_DIR)
print(f'Tổng số ảnh  : {len(X)}')
print(f'Shape X      : {X.shape}  (N, H, W, C)')
print(f'Shape y      : {y.shape}')
print(f'Pixel range  : [{X.min():.2f}, {X.max():.2f}]')

### Visualize một số ảnh mẫu

In [ ]:
samples_per_class = 5
fig, axes = plt.subplots(len(CLASSES), samples_per_class, figsize=(12, 18))
for row, cls in enumerate(CLASSES):
    idxs = np.where(y == row)[0]
    chosen = np.random.choice(idxs, samples_per_class, replace=False)
    for col, idx in enumerate(chosen):
        axes[row, col].imshow(X[idx])
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls, fontsize=10)
plt.tight_layout()
plt.show()

### Chia tập Train / Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {len(X_train)} ảnh')
print(f'Test : {len(X_test)} ảnh')

# Phân phối nhãn
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {CLASSES[u]:12s}: {c} ảnh train')

## 3. Xây dựng kiến trúc CNN

Kiến trúc gồm **3 khối Conv** tăng dần độ phức tạp (32 → 64 → 128 filters), theo sau là **2 lớp Dense** để phân loại.

```
Input (64, 64, 3)
  └─ Conv2D(32, 3×3, ReLU) → MaxPool(2×2) → Dropout(0.25)
  └─ Conv2D(64, 3×3, ReLU) → MaxPool(2×2) → Dropout(0.25)
  └─ Conv2D(128,3×3, ReLU) → MaxPool(2×2) → Dropout(0.25)
  └─ Flatten
  └─ Dense(256, ReLU) → Dropout(0.5)
  └─ Dense(7, Softmax)   ← output
```

**Hàm kích hoạt:**
- **ReLU** (`max(0, x)`) — tất cả lớp ẩn: tránh vanishing gradient, nhanh hội tụ.
- **Softmax** (lớp cuối) — chuyển 7 logit thành 7 xác suất có tổng = 1.

In [ ]:
def build_model(num_classes=7):
    model = tf.keras.Sequential([
        # --- Khối 1: học đặc trưng đơn giản (cạnh, góc) ---
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),

        # --- Khối 2: học đặc trưng trung gian (texture) ---
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),

        # --- Khối 3: học đặc trưng phức tạp (hình dạng hoa) ---
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),

        # --- Phân loại ---
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation='softmax'),
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model(num_classes=len(CLASSES))
model.summary()

**Câu hỏi 1:** Tại sao số filter tăng dần 32 → 64 → 128 qua các khối Conv? Nếu giữ nguyên 32 filter ở cả 3 khối thì điều gì sẽ xảy ra?

$\color{blue}{\textit{Câu trả lời của bạn:}}$ *Điền câu trả lời ở đây*

## 4. Huấn luyện mô hình

Sử dụng 2 callbacks:
- **EarlyStopping** (`patience=10`): dừng sớm nếu `val_loss` không cải thiện sau 10 epoch.
- **ModelCheckpoint**: lưu model có `val_loss` **thấp nhất** trong cả quá trình (không phải epoch cuối).

In [ ]:
MODEL_OUTPUT = 'cnn_flower_model.h5'
EPOCHS       = 50
BATCH_SIZE   = 32

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_OUTPUT, monitor='val_loss', save_best_only=True, verbose=1
    ),
]

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

print(f'\nModel đã lưu tại: {MODEL_OUTPUT}')

### Biểu đồ Loss và Accuracy theo epoch

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['loss'],     label='Train Loss')
ax1.plot(history.history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss theo Epoch')
ax1.legend()

ax2.plot(history.history['accuracy'],     label='Train Accuracy')
ax2.plot(history.history['val_accuracy'], label='Val Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy theo Epoch')
ax2.legend()

plt.tight_layout()
plt.show()

**Câu hỏi 2:** Nhìn vào biểu đồ Loss, nếu `val_loss` bắt đầu tăng trong khi `train_loss` vẫn giảm thì hiện tượng đó gọi là gì? Dropout giúp giải quyết vấn đề này như thế nào?

$\color{blue}{\textit{Câu trả lời của bạn:}}$ *Điền câu trả lời ở đây*

## 5. Đánh giá mô hình

In [ ]:
# Load lại model tốt nhất đã lưu
best_model = tf.keras.models.load_model(MODEL_OUTPUT)

loss, acc = best_model.evaluate(X_test, y_test, verbose=0)
print('=' * 40)
print(f'Test Loss    : {loss:.4f}')
print(f'Test Accuracy: {acc * 100:.2f}%')
print('=' * 40)

# Dự đoán
y_pred_proba = best_model.predict(X_test, verbose=0)
y_pred       = np.argmax(y_pred_proba, axis=1)

### Báo cáo chi tiết từng lớp (Precision / Recall / F1)

In [ ]:
print(classification_report(y_test, y_pred, target_names=CLASSES))

### Ma trận nhầm lẫn (Confusion Matrix)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.ylabel('Nhãn thực tế')
plt.xlabel('Nhãn dự đoán')
plt.title('Confusion Matrix — CNN 7 lớp hoa')
plt.tight_layout()
plt.show()

**Câu hỏi 3:** Nhìn vào Confusion Matrix, lớp nào bị nhận nhầm nhiều nhất? Tại sao theo bạn mô hình lại dễ nhầm 2 lớp đó?

$\color{blue}{\textit{Câu trả lời của bạn:}}$ *Điền câu trả lời ở đây*

## 6. Visualize Feature Maps

Xem model đang "nhìn thấy" gì ở mỗi lớp Conv khi xử lý 1 ảnh hoa.

In [ ]:
# Lấy 1 ảnh test ngẫu nhiên
sample_idx   = np.random.randint(len(X_test))
sample_image = X_test[sample_idx]
sample_label = CLASSES[y_test[sample_idx]]

plt.figure(figsize=(3, 3))
plt.imshow(sample_image)
plt.title(f'Ảnh gốc: {sample_label}')
plt.axis('off')
plt.show()

# Tạo model trích feature maps từ 3 lớp Conv
layer_names   = ['conv2d', 'conv2d_1', 'conv2d_2']
feature_model = tf.keras.Model(
    inputs  = best_model.input,
    outputs = [best_model.get_layer(n).output for n in layer_names]
)

feature_maps = feature_model.predict(sample_image[np.newaxis, ...], verbose=0)

for layer_name, fmap in zip(layer_names, feature_maps):
    n_filters = min(16, fmap.shape[-1])   # hiện tối đa 16 filter
    fig, axes = plt.subplots(2, n_filters // 2, figsize=(16, 4))
    fig.suptitle(f'Feature maps — {layer_name} ({fmap.shape[1]}×{fmap.shape[2]}, {fmap.shape[3]} filters)', fontsize=12)
    for i, ax in enumerate(axes.flat):
        ax.imshow(fmap[0, :, :, i], cmap='viridis')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 7. Thử dự đoán 1 ảnh bất kỳ

In [ ]:
def predict_image(image_path: str, model):
    img = cv2.imread(image_path)
    if img is None:
        print(f'Không đọc được ảnh: {image_path}')
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (64, 64))
    x = img_resized.astype(np.float32) / 255.0

    probs = model.predict(x[np.newaxis, ...], verbose=0)[0]
    pred_idx = int(np.argmax(probs))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(img_rgb)
    ax1.set_title(f'Dự đoán: {CLASSES[pred_idx]} ({probs[pred_idx]*100:.1f}%)')
    ax1.axis('off')

    colors = ['#ef4444' if i == pred_idx else '#6366f1' for i in range(len(CLASSES))]
    ax2.barh(CLASSES, probs, color=colors)
    ax2.set_xlim(0, 1)
    ax2.set_xlabel('Xác suất')
    ax2.set_title('Phân phối xác suất 7 lớp')
    plt.tight_layout()
    plt.show()

# Thay đường dẫn ảnh của bạn vào đây:
# predict_image('path/to/your/flower.jpg', best_model)

## 8. So sánh CNN với các phương pháp truyền thống

| Phương pháp | Extractor | Accuracy (test) | Ghi chú |
|---|---|---|---|
| RandomForest | ORB + BoVW | ~68% | Nhanh, ít dữ liệu |
| XGBoost | ORB + BoVW | ~70.5% | Tốt nhất trong truyền thống |
| SVM (RBF) | ORB + BoVW | ~72% | Hiệu quả với feature thủ công |
| **CNN (3 Conv)** | **Tự học từ pixel** | **~78%** | Không cần feature engineering |

**Tại sao CNN chỉ hơn RF ~10% dù là Deep Learning?**
- Input nhỏ (64×64) mất nhiều detail.
- Không có data augmentation.
- Dataset ~11.200 ảnh còn nhỏ cho CNN.

**Để cải thiện CNN:**
- Tăng input lên 128×128 hoặc 224×224.
- Thêm data augmentation (flip, rotate, zoom, brightness).
- Dùng Transfer Learning: MobileNetV2 / EfficientNetB0 pretrained ImageNet → fine-tune.

**Câu hỏi 4:** Nếu dùng Transfer Learning với MobileNetV2, bạn cần thay đổi gì trong bước tiền xử lý ảnh (input_shape, normalize) và tại sao?

$\color{blue}{\textit{Câu trả lời của bạn:}}$ *Điền câu trả lời ở đây*